Fuzzy clustering of both veg space and optical space

In [3]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score

# =============================================================================
# PATHS
# =============================================================================

csv_path = Path(r"C:\NCA_DATA\Vegetation Data\NCA_Master_Aligned_spp_fg.csv")
out_dir = Path(r"C:\NCA_DATA\Vegetation Data\cluster\veg_cluster_discrete")
out_dir.mkdir(parents=True, exist_ok=True)

sil_plot_png = out_dir / "veg_silhouette.png"
wcss_plot_png = out_dir / "veg_wcss.png"
ch_plot_png = out_dir / "veg_calinski_harabasz.png"

# =============================================================================
# SETTINGS
# =============================================================================

K_MIN = 2
K_MAX = 25
RANDOM_STATE = 42
N_INIT = 20
MAX_ITER = 1000

SIL_SAMPLE_SIZE = 10000   # set None to use all rows
USE_MINIBATCH = False

# exploratory only; change later after inspecting diagnostics
FINAL_K = 16

# =============================================================================
# LOAD DATA
# =============================================================================

print("Reading vegetation dataset...")
df = pd.read_csv(csv_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

# =============================================================================
# IDENTIFY COLUMNS
# =============================================================================

meta_cols = {
    "Plot", "Year",
    "total_cover", "total_cover_fg",
    "SHRUB_total"
}
fg_cols = [c for c in df.columns if c.endswith("_FG")]
species_cols = [c for c in df.columns if c not in meta_cols and c not in fg_cols]

print(f"Detected {len(species_cols)} species/catchall columns.")
print(f"Detected {len(fg_cols)} FG columns.")

# =============================================================================
# BUILD MATRIX
# =============================================================================

X_raw = df[species_cols].copy()

for c in species_cols:
    X_raw[c] = pd.to_numeric(X_raw[c], errors="coerce")

X_raw = X_raw.fillna(0.0)

row_sums = X_raw.sum(axis=1).values
keep = row_sums > 0

if not np.all(keep):
    print(f"Dropping {(~keep).sum():,} rows with zero species total.")

df_work = df.loc[keep].copy().reset_index(drop=True)
X_raw = X_raw.loc[keep].reset_index(drop=True)

# Hellinger transform
print("Applying Hellinger transform...")
row_sums = X_raw.sum(axis=1).values[:, None]
X_prop = X_raw.values / row_sums
X = np.sqrt(X_prop)

print(f"Matrix shape: {X.shape}")

# =============================================================================
# HELPERS
# =============================================================================

def fit_kmeans(X: np.ndarray, k: int, random_state: int):
    if USE_MINIBATCH:
        km = MiniBatchKMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=N_INIT,
            max_iter=MAX_ITER,
            batch_size=8192
        )
    else:
        km = KMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=N_INIT,
            max_iter=MAX_ITER
        )
    labels = km.fit_predict(X)
    return km, labels

def save_line_plot(x, y, xlabel, ylabel, title, out_path):
    plt.figure(figsize=(8, 5))
    plt.plot(x, y, marker="o")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

# =============================================================================
# K DIAGNOSTICS
# =============================================================================

print("Running K diagnostics...")
results = []

n = X.shape[0]
if SIL_SAMPLE_SIZE is not None and n > SIL_SAMPLE_SIZE:
    rng = np.random.default_rng(RANDOM_STATE)
    sil_idx = rng.choice(n, size=SIL_SAMPLE_SIZE, replace=False)
    X_sil = X[sil_idx]
else:
    sil_idx = None
    X_sil = X

for k in range(K_MIN, K_MAX + 1):
    print(f"  k = {k}")
    km, labels = fit_kmeans(X, k, RANDOM_STATE)

    if sil_idx is not None:
        sil = silhouette_score(X_sil, labels[sil_idx], metric="euclidean")
    else:
        sil = silhouette_score(X, labels, metric="euclidean")

    ch = calinski_harabasz_score(X, labels)

    results.append({
        "k": k,
        "wcss": float(km.inertia_),
        "silhouette": float(sil),
        "calinski_harabasz": float(ch)
    })

metrics = pd.DataFrame(results)

# top performers
top_sil = metrics.sort_values("silhouette", ascending=False).head(5)
top_ch = metrics.sort_values("calinski_harabasz", ascending=False).head(5)

print("\nTop K by silhouette:")
print(top_sil)

print("\nTop K by Calinski-Harabasz:")
print(top_ch)

# union of candidates
candidate_k = sorted(set(top_sil["k"]).union(set(top_ch["k"])))
print("\nCandidate K values:", candidate_k)

# =============================================================================
# PLOTS
# =============================================================================

save_line_plot(
    metrics["k"], metrics["silhouette"],
    "k", "Mean silhouette",
    "Vegetation clustering: silhouette",
    sil_plot_png
)

save_line_plot(
    metrics["k"], metrics["wcss"],
    "k", "WCSS",
    "Vegetation clustering: WCSS",
    wcss_plot_png
)

save_line_plot(
    metrics["k"], metrics["calinski_harabasz"],
    "k", "Calinski-Harabasz score",
    "Vegetation clustering: Calinski-Harabasz",
    ch_plot_png
)

print("\nPlots written:")
print(sil_plot_png)
print(wcss_plot_png)
print(ch_plot_png)

# =============================================================================
# CONSOLE OUTPUT
# =============================================================================

print("\nDiagnostics:")
print(metrics.to_string(index=False))

print("\nTop k by silhouette:")
print(metrics.sort_values("silhouette", ascending=False).head(10).to_string(index=False))

print("\nTop k by Calinski-Harabasz:")
print(metrics.sort_values("calinski_harabasz", ascending=False).head(10).to_string(index=False))

# =============================================================================
# OPTIONAL FINAL FIT FOR QUICK LOOK
# =============================================================================

print(f"\nQuick exploratory fit at k = {FINAL_K}...")
km_final, final_labels = fit_kmeans(X, FINAL_K, RANDOM_STATE)

df_work["veg_cluster"] = final_labels + 1

print("\nCluster sizes:")
print(df_work["veg_cluster"].value_counts().sort_index().to_string())

species_summary = (
    df_work.groupby("veg_cluster")[species_cols]
    .mean()
)

print("\nTop 10 species/catchall means by cluster:")
for cluster_id in species_summary.index:
    top = species_summary.loc[cluster_id].sort_values(ascending=False).head(10)
    print(f"\nCluster {cluster_id}")
    print(top.to_string())

if fg_cols:
    fg_summary = (
        df_work.groupby("veg_cluster")[fg_cols]
        .mean()
    )

    print("\nFG means by cluster:")
    print(fg_summary.to_string())


for k in candidate_k:
    print(f"\n=== Inspecting k = {k} ===")

    km, labels = fit_kmeans(X, k, RANDOM_STATE)
    df_work["veg_cluster"] = labels + 1

    # cluster sizes
    sizes = df_work["veg_cluster"].value_counts().sort_index()
    print("\nCluster sizes:")
    print(sizes)

    # FG summaries (much more interpretable than species)
    fg_summary = (
        df_work.groupby("veg_cluster")[fg_cols]
        .mean()
    )

    print("\nFG summary:")
    print(fg_summary.round(2))

Reading vegetation dataset...
Rows: 717
Columns: 86
Detected 71 species/catchall columns.
Detected 10 FG columns.
Applying Hellinger transform...
Matrix shape: (717, 71)
Running K diagnostics...
  k = 2
  k = 3
  k = 4
  k = 5


c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\c

  k = 6
  k = 7
  k = 8


c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(


  k = 9
  k = 10
  k = 11


...


Cluster sizes:
veg_cluster
1     158
2      94
3      13
4      41
5      57
6      43
7      73
8      32
9      75
10     69
11     39
12     16
13      7
Name: count, dtype: int64

FG summary:
             ARTR_FG  BAREGROUND_FG  BIOCRUST_FG  EAG_FG  EF_FG  LITTER_FG  \
veg_cluster                                                                  
1               1.43           8.45         4.00   59.63   6.08       7.08   
2               2.82          29.15        13.25    3.87  11.55      14.79   
3              17.86          24.75        16.20    6.00   6.11      20.22   
4               0.00          25.93        21.24    2.68  29.22      12.76   
5               0.34          17.32         9.86   16.73   1.82      12.57   
6               1.62          12.15         4.50   61.84   5.66       6.62   
7               1.61          18.11        11.54    4.07  10.28       8.79   
8               0.08          24.44         7.41   10.56   0.90      20.04   
9               0.20   

c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(



Cluster sizes:
veg_cluster
1      28
2      30
3      50
4     139
5      41
6      66
7      28
8      46
9      66
10     60
11     16
12      9
13     86
14     29
15     23
Name: count, dtype: int64

FG summary:
             ARTR_FG  BAREGROUND_FG  BIOCRUST_FG  EAG_FG  EF_FG  LITTER_FG  \
veg_cluster                                                                  
1               0.00          63.12         2.18    1.52  15.23       6.62   
2               0.08          22.28         7.75    9.69   0.92      19.66   
3               0.16          17.43         9.80   15.91   1.05      13.01   
4               2.29           6.42         4.17   63.13   3.75       6.87   
5               0.00          25.93        21.24    2.68  29.22      12.76   
6               1.01          20.90         9.86    3.60  10.66       9.47   
7               0.01          36.84        21.31    1.26  19.00      10.55   
8               4.15          22.98         4.84   42.40   1.12       9.14   
9  

c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(



Cluster sizes:
veg_cluster
1      67
2      68
3      25
4      20
5      19
6      10
7     138
8       7
9      57
10     24
11      9
12     13
13     41
14     22
15      9
16     51
17     14
18     92
19     31
Name: count, dtype: int64

FG summary:
             ARTR_FG  BAREGROUND_FG  BIOCRUST_FG  EAG_FG  EF_FG  LITTER_FG  \
veg_cluster                                                                  
1               0.09          17.33         9.69   10.29   1.32      17.12   
2               0.98          18.80        11.34    3.25  10.86       8.40   
3               0.00          54.49         2.92    0.03  11.52      21.50   
4               0.04          19.70         4.80   11.10   1.14      18.30   
5              22.57          17.34         6.45   35.57   0.29      14.25   
6              19.05          26.17        18.34    2.24   4.26      21.05   
7               1.93           6.97         4.16   61.55   4.84       6.98   
8               0.00          69.28      

fuzzy time! cmeans using xie-beni, FPC

In [5]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from scipy.spatial.distance import cdist

# =============================================================================
# PATHS
# =============================================================================

csv_path = Path(r"C:\NCA_DATA\Vegetation Data\NCA_Master_Aligned_spp_fg.csv")
out_dir = Path(r"C:\NCA_DATA\Vegetation Data\cluster\fuzzy_veg")
out_dir.mkdir(parents=True, exist_ok=True)

# =============================================================================
# SETTINGS
# =============================================================================

K_MIN = 2
K_MAX = 25
RANDOM_STATE = 42
N_INIT = 20
MAX_ITER = 300
TOL = 1e-5

# FCM requires m > 1.0
M_LIST = [1.5, 2.0]

# =============================================================================
# LOAD DATA
# =============================================================================

print("Reading vegetation dataset...")
df = pd.read_csv(csv_path)
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

# =============================================================================
# IDENTIFY COLUMNS
# =============================================================================

meta_cols = {
    "Plot", "Year",
    "total_cover", "total_cover_fg",
    "SHRUB_total"
}

fg_cols = [c for c in df.columns if c.endswith("_FG")]

print(f"Detected {len(fg_cols)} FG columns.")

# =============================================================================
# BUILD FG MATRIX
# =============================================================================

X_raw = df[fg_cols].copy()

for c in fg_cols:
    X_raw[c] = pd.to_numeric(X_raw[c], errors="coerce")

X_raw = X_raw.fillna(0.0)

row_sums = X_raw.sum(axis=1).values
keep = row_sums > 0

if not np.all(keep):
    print(f"Dropping {(~keep).sum():,} rows with zero FG total.")

df_work = df.loc[keep].copy().reset_index(drop=True)
X_raw = X_raw.loc[keep].reset_index(drop=True)

# Hellinger transform
print("Applying Hellinger transform to FG matrix...")
row_sums = X_raw.sum(axis=1).values[:, None]
X_prop = X_raw.values / row_sums
X = np.sqrt(X_prop)

print(f"FG matrix shape: {X.shape}")
print("FG columns used:", fg_cols)

# =============================================================================
# HELPERS
# =============================================================================

def fit_kmeans(X: np.ndarray, k: int, random_state: int):
    km = KMeans(
        n_clusters=k,
        random_state=random_state,
        n_init=N_INIT,
        max_iter=1000
    )
    labels = km.fit_predict(X)
    return km, labels

def init_membership(n: int, k: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    U = rng.random((n, k))
    U = U / U.sum(axis=1, keepdims=True)
    return U

def fuzzy_cmeans(
    X: np.ndarray,
    k: int,
    m: float,
    max_iter: int = 300,
    tol: float = 1e-5,
    seed: int = 42
):
    """
    Basic fuzzy c-means.
    X: (n, p)
    Returns:
      centers: (k, p)
      U: (n, k)
      jm: objective history
    """
    if m <= 1.0:
        raise ValueError("FCM requires m > 1.0")

    n = X.shape[0]
    U = init_membership(n, k, seed)
    jm = []

    eps = 1e-12

    for _ in range(max_iter):
        U_old = U.copy()

        Um = U ** m
        centers = (Um.T @ X) / (Um.sum(axis=0)[:, None] + eps)

        D = cdist(X, centers, metric="euclidean")
        D = np.fmax(D, eps)

        # If any point is exactly at a center, give it full membership there
        zero_mask = D <= eps
        if np.any(zero_mask):
            U = np.zeros_like(D)
            row_idx = np.where(zero_mask.any(axis=1))[0]
            for i in row_idx:
                j = np.argmin(D[i])
                U[i, j] = 1.0
            nonzero_rows = np.setdiff1d(np.arange(n), row_idx)
            if len(nonzero_rows) > 0:
                Dnz = D[nonzero_rows]
                power = -2.0 / (m - 1.0)
                tmp = Dnz ** power
                U[nonzero_rows] = tmp / tmp.sum(axis=1, keepdims=True)
        else:
            power = -2.0 / (m - 1.0)
            tmp = D ** power
            U = tmp / tmp.sum(axis=1, keepdims=True)

        obj = np.sum((U ** m) * (D ** 2))
        jm.append(obj)

        if np.max(np.abs(U - U_old)) < tol:
            break

    return centers, U, np.array(jm)

def fuzzy_partition_coefficient(U: np.ndarray) -> float:
    n = U.shape[0]
    return np.sum(U ** 2) / n

def partition_entropy(U: np.ndarray) -> float:
    eps = 1e-12
    return -np.sum(U * np.log(U + eps)) / U.shape[0]

def xie_beni_index(X: np.ndarray, centers: np.ndarray, U: np.ndarray, m: float) -> float:
    eps = 1e-12
    D = cdist(X, centers, metric="euclidean")
    num = np.sum((U ** m) * (D ** 2))

    if centers.shape[0] < 2:
        return np.nan

    center_dist = cdist(centers, centers, metric="euclidean")
    np.fill_diagonal(center_dist, np.inf)
    min_sep_sq = np.min(center_dist) ** 2

    return num / (X.shape[0] * (min_sep_sq + eps))

def save_metric_plot(dfm: pd.DataFrame, metric: str, title: str, out_path: Path):
    plt.figure(figsize=(8, 5))
    for m in sorted(dfm["m"].unique()):
        sub = dfm[dfm["m"] == m].sort_values("k")
        plt.plot(sub["k"], sub[metric], marker="o", label=f"m={m}")
    plt.xlabel("k")
    plt.ylabel(metric)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

# =============================================================================
# HARD K-MEANS REFERENCE (m = 1 LIMIT)
# =============================================================================

print("\nRunning hard k-means reference...")
hard_results = []

for k in range(K_MIN, K_MAX + 1):
    km, labels = fit_kmeans(X, k, RANDOM_STATE)

    sil = silhouette_score(X, labels, metric="euclidean")
    ch = calinski_harabasz_score(X, labels)

    hard_results.append({
        "method": "kmeans",
        "m": 1.0,
        "k": k,
        "silhouette": float(sil),
        "calinski_harabasz": float(ch),
        "wcss": float(km.inertia_)
    })

hard_metrics = pd.DataFrame(hard_results)

print("\nTop hard k by silhouette:")
print(hard_metrics.sort_values("silhouette", ascending=False).head(10).to_string(index=False))

# =============================================================================
# FUZZY C-MEANS SWEEP
# =============================================================================

print("\nRunning fuzzy c-means diagnostics...")
fuzzy_results = []

for m in M_LIST:
    print(f"\n--- m = {m} ---")
    for k in range(K_MIN, K_MAX + 1):
        print(f"  k = {k}")
        centers, U, jm = fuzzy_cmeans(
            X, k, m,
            max_iter=MAX_ITER,
            tol=TOL,
            seed=RANDOM_STATE
        )

        hard_labels = np.argmax(U, axis=1)

        fpc = fuzzy_partition_coefficient(U)
        pe = partition_entropy(U)
        xb = xie_beni_index(X, centers, U, m)

        sil = silhouette_score(X, hard_labels, metric="euclidean")
        ch = calinski_harabasz_score(X, hard_labels)

        fuzzy_results.append({
            "method": "fcm",
            "m": m,
            "k": k,
            "fpc": float(fpc),
            "partition_entropy": float(pe),
            "xie_beni": float(xb),
            "silhouette_hardened": float(sil),
            "calinski_harabasz_hardened": float(ch),
            "objective_final": float(jm[-1]),
            "n_iter": int(len(jm))
        })

fuzzy_metrics = pd.DataFrame(fuzzy_results)

# =============================================================================
# PLOTS
# =============================================================================

save_metric_plot(
    fuzzy_metrics, "fpc",
    "Fuzzy vegetation clustering: FPC (higher better)",
    out_dir / "fcm_fpc.png"
)

save_metric_plot(
    fuzzy_metrics, "partition_entropy",
    "Fuzzy vegetation clustering: partition entropy (lower better)",
    out_dir / "fcm_partition_entropy.png"
)

save_metric_plot(
    fuzzy_metrics, "xie_beni",
    "Fuzzy vegetation clustering: Xie-Beni (lower better)",
    out_dir / "fcm_xie_beni.png"
)

save_metric_plot(
    fuzzy_metrics, "silhouette_hardened",
    "Fuzzy vegetation clustering: hardened silhouette",
    out_dir / "fcm_silhouette_hardened.png"
)

print("\nPlots written to:")
for fn in [
    "fcm_fpc.png",
    "fcm_partition_entropy.png",
    "fcm_xie_beni.png",
    "fcm_silhouette_hardened.png"
]:
    print(out_dir / fn)

# =============================================================================
# CONSOLE SUMMARY
# =============================================================================

print("\nTop FCM by FPC:")
print(
    fuzzy_metrics.sort_values(["fpc", "xie_beni"], ascending=[False, True])
    .head(12)
    .to_string(index=False)
)

print("\nTop FCM by lowest Xie-Beni:")
print(
    fuzzy_metrics.sort_values("xie_beni", ascending=True)
    .head(12)
    .to_string(index=False)
)

print("\nTop FCM by lowest partition entropy:")
print(
    fuzzy_metrics.sort_values("partition_entropy", ascending=True)
    .head(12)
    .to_string(index=False)
)

# =============================================================================
# PICK BEST MODEL FOR INSPECTION
# =============================================================================
# Simple rule:
#   - prioritize low Xie-Beni
#   - among those, prefer higher FPC
#   - and avoid extreme k if several are similar
# You can change this after inspecting the curves.

best_row = (
    fuzzy_metrics.sort_values(
        ["xie_beni", "fpc", "partition_entropy"],
        ascending=[True, False, True]
    )
    .iloc[0]
)

BEST_M = float(best_row["m"])
BEST_K = int(best_row["k"])

print(f"\nSelected exploratory best model: m = {BEST_M}, k = {BEST_K}")

centers, U, jm = fuzzy_cmeans(
    X, BEST_K, BEST_M,
    max_iter=MAX_ITER,
    tol=TOL,
    seed=RANDOM_STATE
)

hard_labels = np.argmax(U, axis=1) + 1
max_membership = np.max(U, axis=1)

df_inspect = df_work.copy()
df_inspect["veg_cluster_fuzzy"] = hard_labels
df_inspect["max_membership"] = max_membership

print("\nCluster sizes from hardened fuzzy labels:")
print(df_inspect["veg_cluster_fuzzy"].value_counts().sort_index().to_string())

if fg_cols:
    fg_summary = (
        df_inspect.groupby("veg_cluster_fuzzy")[fg_cols]
        .mean()
        .round(2)
    )

    print("\nFG summary for selected fuzzy model:")
    print(fg_summary.to_string())

print("\nMembership concentration summary:")
print(pd.Series(max_membership).describe().to_string())

Reading vegetation dataset...
Rows: 717
Columns: 85
Detected 10 FG columns.
Applying Hellinger transform to FG matrix...
FG matrix shape: (717, 10)
FG columns used: ['ARTR_FG', 'BAREGROUND_FG', 'BIOCRUST_FG', 'EAG_FG', 'EF_FG', 'LITTER_FG', 'NPF_FG', 'OTHER_FG', 'PBG_FG', 'SHRUB_FG']

Running hard k-means reference...


c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\c


Top hard k by silhouette:
method   m  k  silhouette  calinski_harabasz       wcss
kmeans 1.0  6    0.251602         164.842500 140.671911
kmeans 1.0  5    0.247248         173.884666 153.647658
kmeans 1.0  7    0.228527         153.089495 132.424094
kmeans 1.0  4    0.224257         176.593838 174.261332
kmeans 1.0  2    0.215579         221.150997 231.988481
kmeans 1.0 10    0.214237         131.711064 113.478309
kmeans 1.0  9    0.212817         136.501201 119.471610
kmeans 1.0  8    0.211327         143.541208 125.659494
kmeans 1.0  3    0.208153         184.459215 200.266704
kmeans 1.0 12    0.202277         122.732764 104.200741

Running fuzzy c-means diagnostics...

--- m = 1.5 ---
  k = 2
  k = 3
  k = 4
  k = 5
  k = 6
  k = 7
  k = 8
  k = 9
  k = 10
  k = 11
  k = 12
  k = 13
  k = 14
  k = 15
  k = 16
  k = 17
  k = 18
  k = 19
  k = 20
  k = 21
  k = 22
  k = 23
  k = 24
  k = 25

--- m = 2.0 ---
  k = 2
  k = 3
  k = 4
  k = 5
  k = 6
  k = 7
  k = 8
  k = 9
  k = 10
  k 

Optical fuzzy below

In [7]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler

# =============================================================================
# PATHS
# =============================================================================

base_dir = Path(r"C:\NCA_DATA\Clustering\cluster_v3")
energy_file = base_dir / "X_energy_1M_clipped.npy"

out_dir = Path(r"C:\NCA_DATA\Clustering\cluster_v4_energy\fuzzy_energy")
out_dir.mkdir(parents=True, exist_ok=True)

# =============================================================================
# SETTINGS
# =============================================================================

K_VALUES = list(range(10, 21))
M_LIST = [1.5, 2.0]

RANDOM_STATE = 42
N_INIT = 20
MAX_ITER = 300
TOL = 1e-5

# subsample for silhouette / PCA plotting to keep it fast
SIL_N = 40000
PLOT_N = 40000

# =============================================================================
# HELPERS
# =============================================================================

def fit_kmeans(X: np.ndarray, k: int, random_state: int):
    km = KMeans(
        n_clusters=k,
        random_state=random_state,
        n_init=N_INIT,
        max_iter=1000
    )
    labels = km.fit_predict(X)
    return km, labels

def init_membership(n: int, k: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    U = rng.random((n, k))
    U = U / U.sum(axis=1, keepdims=True)
    return U

def fuzzy_cmeans(
    X: np.ndarray,
    k: int,
    m: float,
    max_iter: int = 300,
    tol: float = 1e-5,
    seed: int = 42
):
    if m <= 1.0:
        raise ValueError("FCM requires m > 1.0")

    n = X.shape[0]
    U = init_membership(n, k, seed)
    jm = []
    eps = 1e-12

    for _ in range(max_iter):
        U_old = U.copy()

        Um = U ** m
        centers = (Um.T @ X) / (Um.sum(axis=0)[:, None] + eps)

        D = cdist(X, centers, metric="euclidean")
        D = np.fmax(D, eps)

        power = -2.0 / (m - 1.0)
        tmp = D ** power
        U = tmp / tmp.sum(axis=1, keepdims=True)

        obj = np.sum((U ** m) * (D ** 2))
        jm.append(obj)

        if np.max(np.abs(U - U_old)) < tol:
            break

    return centers, U, np.array(jm)

def fuzzy_partition_coefficient(U: np.ndarray) -> float:
    return np.sum(U ** 2) / U.shape[0]

def partition_entropy(U: np.ndarray) -> float:
    eps = 1e-12
    return -np.sum(U * np.log(U + eps)) / U.shape[0]

def xie_beni_index(X: np.ndarray, centers: np.ndarray, U: np.ndarray, m: float) -> float:
    eps = 1e-12
    D = cdist(X, centers, metric="euclidean")
    num = np.sum((U ** m) * (D ** 2))

    center_dist = cdist(centers, centers, metric="euclidean")
    np.fill_diagonal(center_dist, np.inf)
    min_sep_sq = np.min(center_dist) ** 2

    return num / (X.shape[0] * (min_sep_sq + eps))

def compute_assignment_margin(X: np.ndarray, centers: np.ndarray):
    D = cdist(X, centers, metric="euclidean")
    two_smallest = np.partition(D, kth=1, axis=1)[:, :2]
    two_smallest.sort(axis=1)

    d1 = two_smallest[:, 0]
    d2 = two_smallest[:, 1]

    denom = np.where(d2 <= 1e-12, 1e-12, d2)
    margin = (d2 - d1) / denom
    ratio = d1 / denom

    return {
        "margin_mean": float(np.mean(margin)),
        "margin_sd": float(np.std(margin)),
        "margin_median": float(np.median(margin)),
        "d1_over_d2_mean": float(np.mean(ratio)),
        "d1_over_d2_median": float(np.median(ratio)),
        "prop_ambiguous_0p90": float(np.mean(ratio > 0.90)),
        "prop_ambiguous_0p95": float(np.mean(ratio > 0.95)),
    }, margin, ratio

def summarize_cluster_sizes(labels: np.ndarray) -> pd.Series:
    return pd.Series(labels).value_counts().sort_index()

def save_metric_plot(dfm: pd.DataFrame, metric: str, title: str, out_path: Path):
    plt.figure(figsize=(8, 5))
    for m in sorted(dfm["m"].unique()):
        sub = dfm[dfm["m"] == m].sort_values("k")
        plt.plot(sub["k"], sub[metric], marker="o", label=f"m={m}")
    plt.xlabel("k")
    plt.ylabel(metric)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

def save_pca_scatter(X: np.ndarray, labels: np.ndarray, centers: np.ndarray, out_path: Path, title: str, n_plot: int = 40000):
    rng = np.random.default_rng(RANDOM_STATE)
    idx = rng.choice(X.shape[0], size=min(n_plot, X.shape[0]), replace=False)

    X_sub = X[idx]
    lab_sub = labels[idx]

    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X2 = pca.fit_transform(X_sub)
    C2 = pca.transform(centers)

    plt.figure(figsize=(8, 6))
    plt.scatter(X2[:, 0], X2[:, 1], c=lab_sub, s=3, alpha=0.4, cmap="tab20")
    plt.scatter(C2[:, 0], C2[:, 1], c="black", s=80, marker="x")
    plt.title(title)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

# =============================================================================
# LOAD + STANDARDIZE
# =============================================================================

print("Loading energy matrix...")
X = np.load(energy_file)
print("Shape:", X.shape)

print("Standardizing...")
scaler = StandardScaler()
Xz = scaler.fit_transform(X)

# optional feature names if you have them; otherwise generic names
feature_names = [f"feat_{i}" for i in range(Xz.shape[1])]

# subsample for silhouette if needed
if Xz.shape[0] > SIL_N:
    rng = np.random.default_rng(RANDOM_STATE)
    sil_idx = rng.choice(Xz.shape[0], size=SIL_N, replace=False)
    X_sil = Xz[sil_idx]
else:
    sil_idx = None
    X_sil = Xz

# =============================================================================
# HARD K-MEANS REFERENCE
# =============================================================================

print("\nRunning hard k-means reference...")
hard_results = []

for k in K_VALUES:
    print(f"  k = {k}")
    km, labels = fit_kmeans(Xz, k, RANDOM_STATE)

    if sil_idx is not None:
        sil = silhouette_score(X_sil, labels[sil_idx], metric="euclidean")
    else:
        sil = silhouette_score(Xz, labels, metric="euclidean")

    ch = calinski_harabasz_score(Xz, labels)
    margin_stats, _, _ = compute_assignment_margin(Xz, km.cluster_centers_)

    hard_results.append({
        "method": "kmeans",
        "m": 1.0,
        "k": k,
        "silhouette": float(sil),
        "calinski_harabasz": float(ch),
        "wcss": float(km.inertia_),
        **margin_stats
    })

hard_metrics = pd.DataFrame(hard_results)
hard_metrics.to_csv(out_dir / "hard_kmeans_reference.csv", index=False)

print("\nTop hard k by silhouette:")
print(hard_metrics.sort_values("silhouette", ascending=False).to_string(index=False))

# =============================================================================
# FUZZY C-MEANS SWEEP
# =============================================================================

print("\nRunning fuzzy c-means sweep...")
fuzzy_results = []
model_store = {}

for m in M_LIST:
    print(f"\n--- m = {m} ---")
    for k in K_VALUES:
        print(f"  k = {k}")
        centers, U, jm = fuzzy_cmeans(
            Xz, k, m,
            max_iter=MAX_ITER,
            tol=TOL,
            seed=RANDOM_STATE
        )

        hard_labels = np.argmax(U, axis=1)

        fpc = fuzzy_partition_coefficient(U)
        pe = partition_entropy(U)
        xb = xie_beni_index(Xz, centers, U, m)

        if sil_idx is not None:
            sil = silhouette_score(X_sil, hard_labels[sil_idx], metric="euclidean")
        else:
            sil = silhouette_score(Xz, hard_labels, metric="euclidean")

        ch = calinski_harabasz_score(Xz, hard_labels)
        margin_stats, margin_vals, ratio_vals = compute_assignment_margin(Xz, centers)
        max_membership = np.max(U, axis=1)

        fuzzy_results.append({
            "method": "fcm",
            "m": m,
            "k": k,
            "fpc": float(fpc),
            "partition_entropy": float(pe),
            "xie_beni": float(xb),
            "silhouette_hardened": float(sil),
            "calinski_harabasz_hardened": float(ch),
            "objective_final": float(jm[-1]),
            "n_iter": int(len(jm)),
            "mean_max_membership": float(np.mean(max_membership)),
            "median_max_membership": float(np.median(max_membership)),
            **margin_stats
        })

        model_store[(m, k)] = {
            "centers": centers,
            "U": U,
            "hard_labels": hard_labels,
            "max_membership": max_membership,
            "margin": margin_vals,
            "ratio": ratio_vals
        }

fuzzy_metrics = pd.DataFrame(fuzzy_results)
fuzzy_metrics.to_csv(out_dir / "fcm_metrics.csv", index=False)

# =============================================================================
# PLOTS
# =============================================================================

save_metric_plot(
    fuzzy_metrics, "fpc",
    "Optical FCM: FPC (higher better)",
    out_dir / "fcm_fpc.png"
)

save_metric_plot(
    fuzzy_metrics, "partition_entropy",
    "Optical FCM: partition entropy (lower better)",
    out_dir / "fcm_partition_entropy.png"
)

save_metric_plot(
    fuzzy_metrics, "xie_beni",
    "Optical FCM: Xie-Beni (lower better)",
    out_dir / "fcm_xie_beni.png"
)

save_metric_plot(
    fuzzy_metrics, "silhouette_hardened",
    "Optical FCM: hardened silhouette",
    out_dir / "fcm_silhouette_hardened.png"
)

save_metric_plot(
    fuzzy_metrics, "mean_max_membership",
    "Optical FCM: mean max membership",
    out_dir / "fcm_mean_max_membership.png"
)

# =============================================================================
# SUMMARIES
# =============================================================================

print("\nTop FCM by FPC:")
print(
    fuzzy_metrics.sort_values(["fpc", "xie_beni"], ascending=[False, True])
    .head(12)
    .to_string(index=False)
)

print("\nTop FCM by lowest Xie-Beni:")
print(
    fuzzy_metrics.sort_values("xie_beni", ascending=True)
    .head(12)
    .to_string(index=False)
)

print("\nTop FCM by hardened silhouette:")
print(
    fuzzy_metrics.sort_values("silhouette_hardened", ascending=False)
    .head(12)
    .to_string(index=False)
)

# =============================================================================
# INSPECT A FEW MODELS
# =============================================================================

# choose top 3 by hardened silhouette for inspection
inspect_rows = fuzzy_metrics.sort_values("silhouette_hardened", ascending=False).head(3)

summary_rows = []
centroid_rows = []

for _, row in inspect_rows.iterrows():
    m = float(row["m"])
    k = int(row["k"])
    key = (m, k)

    centers = model_store[key]["centers"]
    hard_labels = model_store[key]["hard_labels"]
    max_membership = model_store[key]["max_membership"]

    sizes = summarize_cluster_sizes(hard_labels + 1)

    print(f"\n=== Inspecting m={m}, k={k} ===")
    print("\nCluster sizes:")
    print(sizes.to_string())

    print("\nMembership concentration:")
    print(pd.Series(max_membership).describe().to_string())

    # centroid feature summary
    centroid_df = pd.DataFrame(centers, columns=feature_names)
    centroid_df.insert(0, "cluster", np.arange(1, k + 1))
    centroid_df.insert(0, "k", k)
    centroid_df.insert(0, "m", m)
    centroid_rows.append(centroid_df)

    # simple cluster summary table
    for cl, sz in sizes.items():
        summary_rows.append({
            "m": m,
            "k": k,
            "cluster": cl,
            "n": int(sz),
            "mean_max_membership_all": float(np.mean(max_membership))
        })

    # PCA visualization
    save_pca_scatter(
        Xz,
        hard_labels,
        centers,
        out_dir / f"pca_scatter_m{str(m).replace('.', 'p')}_k{k}.png",
        title=f"Optical FCM hardened labels in PCA space (m={m}, k={k})",
        n_plot=PLOT_N
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(out_dir / "inspected_cluster_sizes.csv", index=False)

centroid_summary_df = pd.concat(centroid_rows, ignore_index=True)
centroid_summary_df.to_csv(out_dir / "inspected_centroid_summary.csv", index=False)

print("\nWritten:")
print(out_dir / "hard_kmeans_reference.csv")
print(out_dir / "fcm_metrics.csv")
print(out_dir / "inspected_cluster_sizes.csv")
print(out_dir / "inspected_centroid_summary.csv")
print("\nPlots:")
for p in sorted(out_dir.glob("*.png")):
    print(p)

Loading energy matrix...
Shape: (999668, 20)
Standardizing...

Running hard k-means reference...
  k = 10
  k = 11
  k = 12
  k = 13
  k = 14
  k = 15
  k = 16
  k = 17
  k = 18
  k = 19
  k = 20

Top hard k by silhouette:
method   m  k  silhouette  calinski_harabasz        wcss  margin_mean  margin_sd  margin_median  d1_over_d2_mean  d1_over_d2_median  prop_ambiguous_0p90  prop_ambiguous_0p95
kmeans 1.0 11    0.328975       1021764.5000 1781746.500     0.467466   0.224608       0.508553         0.532534           0.491447             0.078872             0.038593
kmeans 1.0 10    0.323527       1024426.9375 1955716.500     0.462940   0.224338       0.502772         0.537060           0.497228             0.081661             0.039786
kmeans 1.0 14    0.310416        979345.3750 1455555.750     0.450932   0.223543       0.482531         0.549068           0.517469             0.084110             0.041208
kmeans 1.0 12    0.300517       1003659.5625 1660019.000     0.441993   0.224255 